In [1]:
!pip install opencv-python mlflow scikit-learn pyngrok faiss-cpu

In [2]:
import pandas as pd
import numpy as np
import os
import mlflow
import mlflow.sklearn
from pyngrok import ngrok
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import *
from sklearn.metrics import *
from sklearn.decomposition import PCA
from sklearn.base import BaseEstimator
import matplotlib.pyplot as plt
import cv2
import requests
import faiss
import joblib

In [3]:
class BaselinePCA(BaseEstimator):
    def __init__(self, n_components=0.95, image_size=(64, 64)):
        self.n_components = n_components
        self.image_size = image_size

        self.scaler = StandardScaler()
        self.pca = PCA(
            n_components=self.n_components,
            whiten=True,
            svd_solver='full',
            random_state=42
        )

    def _preprocess(self, img):
        img = cv2.resize(img, self.image_size)
        img = img.astype("float32") / 255.0
        return img.flatten()

    def fit(self, X, paths=None):
        X_proc = np.array([self._preprocess(img) for img in X])

        self.paths_ = paths

        X_scaled = self.scaler.fit_transform(X_proc)
        X_pca = self.pca.fit_transform(X_scaled).astype("float32")

        # normalize
        norms = np.linalg.norm(X_pca, axis=1, keepdims=True)
        X_norm = X_pca / norms

        self.embeddings_ = X_norm.astype("float32")

        # FAISS index
        dim = self.embeddings_.shape[1]
        self.index = faiss.IndexFlatIP(dim)
        self.index.add(self.embeddings_)

        return self

    def recommend(self, query_img, top_k=5):
        q = self._preprocess(query_img).reshape(1, -1)

        q_scaled = self.scaler.transform(q)
        q_pca = self.pca.transform(q_scaled).astype("float32")

        q_norm = q_pca / np.linalg.norm(q_pca, axis=1, keepdims=True)

        scores, idxs = self.index.search(q_norm, top_k)

        return idxs[0], scores[0]

In [4]:
def load_images(folder):
    images = []
    paths = []

    for fname in os.listdir(folder):
        path = os.path.join(folder, fname)

        img = cv2.imread(path)

        if img is not None:
            images.append(img)
            paths.append(path)

    return images, paths

In [ ]:
dataset_path = "/content/drive/MyDrive/image_dataset/Image_Data"

images, paths = load_images(dataset_path)

print(f"Loaded {len(images)} images")

In [ ]:
model = BaselinePCA()
model.fit(images, paths)

# store paths instead of raw images (better)
model.paths_ = paths

In [ ]:
query_img = images[21]

idxs, scores = model.recommend(query_img, top_k=20)

recommended_paths = [model.paths_[i] for i in idxs]

In [ ]:
def show(img, title=""):
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.title(title)
    plt.axis("off")

plt.figure(figsize=(12,4))

# Query
plt.subplot(1, 6, 1)
show(query_img, "Query")

# Results: Display only the first 5 recommended images to fit the 1x6 grid.
for i, idx in enumerate(idxs[:5]):
    img = cv2.imread(model.paths_[idx])
    plt.subplot(1, 6, i+2)
    show(img, f"{scores[i]:.2f}")

plt.show()

## Saving The whole thing

In [ ]:
joblib.dump(model, "baseline_pca.pkl")